# ILRC v2 on MEVID-SC: fair comparison + a usable guarantee (CPU only)

Follows the first ILRC run, where the point thresholds broke their promise (96% promised, 78% delivered) and the strict guarantee merged nothing.

- **Cell 3:** builds global IDs at every merge threshold, for both backbones (CSCI, the OSNet baseline), both scores (similarity, light verifier) and both directions. Nothing is chosen yet.
- **Cell 4, fair comparison (diagnostic):** at the **same delivered precision** (90%, 95%), which score gives fewer IDs per person? It uses the best threshold on the test group itself, so it measures score quality, not a deployable rule. Also plots IDs per person against precision.
- **Cell 5, ILRC v2:** controls the **average wrong-link share per person** with a **person-level bootstrap bound** (90% confidence), chosen on the calibration group only by fixed-sequence testing, then frozen and tested. Run at 5% and 10%.

Runs on a normal CPU runtime in about 5 to 15 minutes. Results go to `MEVID-results/ILRC-v2-SameClothes`.

In [ ]:
# 1. Load both backbones' features on MEVID-SC + the frozen person groups
import os, glob, json, time, numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return os.path.dirname(h[0])
    raise FileNotFoundError(pats[0])

CS_DIR = find([f'{SD}/Data/MEVID-results/CSCI-SameClothes/'
               'test_templates_csci_sc.npz',
               f'{SD}/**/CSCI-SameClothes/test_templates_csci_sc.npz'])
TR_DIR = find([f'{SD}/Data/MEVID-results/Tracker-SameClothes/'
               'test_templates_tracker_sc.npz',
               f'{SD}/**/Tracker-SameClothes/test_templates_tracker_sc.npz'])
OUT = os.path.join(os.path.dirname(CS_DIR), 'ILRC-v2-SameClothes')
os.makedirs(OUT, exist_ok=True)

z = np.load(f'{CS_DIR}/test_templates_csci_sc.npz')
pid, cam = z['pid'], z['cam']
zt = np.load(f'{TR_DIR}/test_templates_tracker_sc.npz')
assert (zt['pid'] == pid).all() and (zt['cam'] == cam).all()
BACKBONES = {'CSCI': z['templates'], 'OSNet baseline (OSNet)': zt['zs']}
for k, v in BACKBONES.items():
    BACKBONES[k] = v / np.maximum(np.linalg.norm(v, axis=1,
                                                 keepdims=True), 1e-8)
proto = json.load(open(f'{CS_DIR}/PROTOCOL_frozen_sc.json'))
fold = np.array([proto['folds'][str(int(p))] for p in pid])

# frozen settings (fixed before any result of this notebook)
SETTINGS = {'pair_precision_target': 0.95,       # baseline rule
            'person_error_limit': 0.05,           # >5% wrong links = fail
            'epsilon': 0.10,                      # max share of people
            'delta': 0.10,                        # failing, 90% confidence
            'k1': 20, 'k2': 6, 'lambda': 0.3,     # k-reciprocal
            'margin_topk': 50, 'group_k': 5, 'seed': 2027}
with open(f'{OUT}/SETTINGS_frozen.json', 'w') as f:
    json.dump(SETTINGS, f, indent=2)
print('Backbones:', list(BACKBONES), '| clips:', len(pid),
      '| people per group:',
      {g: len(np.unique(pid[fold == g])) for g in 'AB'})
print('Results go to', OUT)


In [ ]:
# 2. The light verifier: three checks on the similarity table, no model
def similarity(T):
    S = np.clip(T @ T.T, -1, 1)
    return S


def kreciprocal_jaccard(S, k1, k2, lam):
    """k-reciprocal re-ranking (Zhong et al., CVPR 2017)."""
    n = len(S)
    D = 2 - 2 * S
    D = D / D.max()
    rank = np.argsort(D, axis=1)
    V = np.zeros((n, n), np.float32)
    h = int(round(k1 / 2))
    for i in range(n):
        fwd = rank[i, :k1 + 1]
        bwd = rank[fwd, :k1 + 1]
        kr = fwd[np.where(bwd == i)[0]]
        exp = kr
        for c in kr:
            cf = rank[c, :h + 1]
            cb = rank[cf, :h + 1]
            ckr = cf[np.where(cb == c)[0]]
            if len(np.intersect1d(ckr, kr)) > 2 / 3 * len(ckr):
                exp = np.append(exp, ckr)
        exp = np.unique(exp)
        w = np.exp(-D[i, exp])
        V[i, exp] = w / w.sum()
    if k2 > 1:
        V = np.stack([V[rank[i, :k2]].mean(0) for i in range(n)])
    J = np.zeros((n, n), np.float32)
    for i in range(n):
        nz = np.flatnonzero(V[i])
        m = np.minimum(V[i, nz][None, :], V[:, nz]).sum(1)
        J[i] = 1 - m / (2 - m)
    J = (J + J.T) / 2
    return (1 - lam) * J + lam * D


def identity_margin(S, tau, topk):
    """s(i,j) minus the best similarity of i to a clip NOT linked to j
    (linked = similarity to j >= tau): the identity-level margin."""
    n = len(S)
    M = np.zeros((n, n), np.float32)
    for i in range(n):
        order = np.argsort(-S[i])
        cand = order[order != i][:topk]
        free = S[cand] < tau                     # topk x n: unlinked to j
        first = np.where(free.any(0), free.argmax(0), topk - 1)
        M[i] = S[i] - S[i, cand[first]]
    return (M + M.T) / 2


def group_similarity(S, k):
    """mean similarity of i to j's k nearest clips (set-to-set)."""
    nn = np.argsort(-S, axis=1)[:, :k]
    G = np.zeros_like(S)
    for j in range(len(S)):
        G[:, j] = S[:, nn[j]].mean(1)
    return (G + G.T) / 2


def features(T):
    S = similarity(T)
    st = SETTINGS
    R = np.argsort(np.argsort(-S, axis=1), axis=1)
    feats = {'similarity': S,
             'reranked distance': kreciprocal_jaccard(
                 S, st['k1'], st['k2'], st['lambda']),
             'mutual rank (log)': np.log1p(np.maximum(R, R.T)),
             'group similarity': group_similarity(S, st['group_k'])}
    return feats


def add_margin(feats, tau):
    feats['identity margin'] = identity_margin(
        feats['similarity'], tau, SETTINGS['margin_topk'])
    return feats


def pair_table(feats, idx):
    iu = np.triu_indices(len(idx), 1)
    return np.stack([F[np.ix_(idx, idx)][iu] for F in feats.values()], 1)


In [ ]:
# 3. Build global IDs at every merge threshold, for every backbone,
#    score and direction; store the full sweep (nothing is chosen yet)
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


def person_losses(pred, p):
    """per person: share of its same-ID pairs that are other people."""
    same_c = pred[:, None] == pred[None, :]
    np.fill_diagonal(same_c, False)
    out = []
    for q in np.unique(p):
        rows = same_c[p == q]
        tot = rows.sum()
        out.append(rows[:, p != q].sum() / tot if tot else 0.0)
    return np.array(out)


def score(pred, p, c):
    P = {x: i for i, x in enumerate(np.unique(p))}
    C = {x: j for j, x in enumerate(np.unique(pred))}
    M = np.zeros((len(P), len(C)))
    for a, b in zip(p, pred):
        M[P[a], C[b]] += 1
    r, cc = linear_sum_assignment(-M)
    sp = p[:, None] == p[None, :]; sc = pred[:, None] == pred[None, :]
    off = ~np.eye(len(p), dtype=bool); tp = (sp & sc & off).sum()
    xc = sp & (c[:, None] != c[None, :])
    lp = person_losses(pred, p)
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'people per ID': (M > 0).sum(0).mean(),
            'pair precision %': (100 * tp / (sc & off).sum()
                                 if (sc & off).sum() else np.nan),
            'cross-camera recall %': 100 * (xc & sc).sum() / max(1, xc.sum()),
            'mean wrong-link share %': 100 * lp.mean(),
            'people failing %':
                100 * (lp > SETTINGS['person_error_limit']).mean()}


GRID = np.round(np.arange(0.01, 0.951, 0.01), 2)   # merge distances
SWEEP = []                     # one row per (case, threshold)
CAL_LOSS = {}                  # (case, threshold) -> calibration losses
for bb, T in BACKBONES.items():
    t0 = time.time()
    FE = {g: features(T[fold == g]) for g in 'AB'}
    for cal, ev in [('A', 'B'), ('B', 'A')]:
        ic, ie = np.flatnonzero(fold == cal), np.flatnonzero(fold == ev)
        Sc, Se = FE[cal]['similarity'], FE[ev]['similarity']
        diff = pid[ic][:, None] != pid[ic][None, :]
        tau = float(np.quantile(Sc[diff], 0.95))
        f2c = add_margin(dict(FE[cal]), tau)
        f2e = add_margin(dict(FE[ev]), tau)
        Xc = pair_table(f2c, np.arange(len(ic)))
        Xe = pair_table(f2e, np.arange(len(ie)))
        iu = np.triu_indices(len(ic), 1)
        yc = (pid[ic][:, None] == pid[ic][None, :])[iu]
        lr = make_pipeline(StandardScaler(),
                           LogisticRegression(max_iter=2000)).fit(Xc, yc)

        def pmat(X, n):
            Pm = np.zeros((n, n))
            Pm[np.triu_indices(n, 1)] = lr.predict_proba(X)[:, 1]
            Pm = Pm + Pm.T
            np.fill_diagonal(Pm, 1)
            return Pm

        dists = {'similarity': (1 - Sc, 1 - Se),
                 'light verifier': (1 - pmat(Xc, len(ic)),
                                    1 - pmat(Xe, len(ie)))}
        for sname, (Dc, De) in dists.items():
            Zc = linkage(Dc[np.triu_indices(len(ic), 1)], 'average')
            Ze = linkage(De[np.triu_indices(len(ie), 1)], 'average')
            case = (bb, sname, f'{cal}->{ev}')
            for d in GRID:
                pc = fcluster(Zc, d, 'distance')
                pe = fcluster(Ze, d, 'distance')
                CAL_LOSS[case + (d,)] = person_losses(pc, pid[ic])
                cs = score(pc, pid[ic], cam[ic])
                es = score(pe, pid[ie], cam[ie])
                SWEEP.append({'backbone': bb, 'score': sname,
                              'direction': f'{cal}->{ev}', 'distance': d,
                              'cal pair precision %': cs['pair precision %'],
                              **es})
    print(f'{bb}: swept {len(GRID)} thresholds x 4 cases in '
          f'{time.time() - t0:.0f} s')
SW = pd.DataFrame(SWEEP)
SW.to_csv(f'{OUT}/sweep_all_thresholds.csv', index=False)
print('Saved the full sweep to', OUT)


In [ ]:
# 4. FAIR COMPARISON (diagnostic, not deployable): at the SAME delivered
#    precision on the test group, which score gives fewer IDs per person?
#    Uses the best threshold on the test group itself, only to compare
#    the quality of the two scores independent of calibration.
import matplotlib.pyplot as plt
rows = []
for (bb, sname, dv), g in SW.groupby(['backbone', 'score', 'direction'],
                                     sort=False):
    for lvl in [90, 95]:
        ok = g[g['pair precision %'] >= lvl]
        if len(ok):
            b = ok.loc[ok['IDF1 %'].idxmax()]
            rows.append({'backbone': bb, 'score': sname, 'direction': dv,
                         'precision level %': lvl, 'IDF1 %': b['IDF1 %'],
                         'IDs per person': b['IDs per person'],
                         'cross-camera recall %':
                             b['cross-camera recall %']})
fair = (pd.DataFrame(rows)
        .groupby(['backbone', 'precision level %', 'score'], sort=False)
        [['IDF1 %', 'IDs per person', 'cross-camera recall %']]
        .mean().round(1))
fair.to_csv(f'{OUT}/fair_comparison_matched_precision.csv')
print('AT MATCHED PRECISION (mean of both directions)')
display(fair)

fig, axes = plt.subplots(1, len(BACKBONES), figsize=(12, 4.5))
for ax, bb in zip(np.atleast_1d(axes), BACKBONES):
    for (sname, dv), g in SW[SW.backbone == bb].groupby(['score',
                                                         'direction']):
        g = g.dropna(subset=['pair precision %'])
        ax.plot(g['pair precision %'], g['IDs per person'],
                '-' if sname == 'light verifier' else '--',
                label=f'{sname} ({dv})')
    ax.set_title(bb); ax.set_xlabel('delivered pair precision %')
    ax.set_ylabel('IDs per person (lower is better)')
    ax.set_xlim(70, 100.5); ax.set_ylim(0.9, 12); ax.grid(alpha=.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUT}/curve_ids_per_person_vs_precision.png', dpi=150)
plt.show()


In [ ]:
# 5. ILRC v2: control the AVERAGE wrong-link share per person, with a
#    person-level bootstrap bound (practical with 27 people). Chosen on
#    the calibration group only, by fixed-sequence testing (strict ->
#    permissive, stop at first failure), then frozen and tested.
B = 2000
rng = np.random.default_rng(SETTINGS['seed'])
rows = []
for alpha in [0.05, 0.10]:
    for (bb, sname, dv), g in SW.groupby(['backbone', 'score',
                                          'direction'], sort=False):
        case, best, bound = (bb, sname, dv), None, None
        n = len(CAL_LOSS[case + (GRID[0],)])
        idx = rng.integers(0, n, size=(B, n))
        for d in GRID:
            l = CAL_LOSS[case + (d,)]
            ucb = np.quantile(l[idx].mean(1), 1 - SETTINGS['delta'])
            if ucb > alpha:
                break
            best, bound = d, ucb
        if best is not None:
            r = g[g.distance == best].iloc[0]
        else:                                    # nothing passes: no merges
            ie = np.flatnonzero(fold == dv[-1])
            r = pd.Series(score(np.arange(len(ie)), pid[ie], cam[ie]))
        rows.append({'alpha %': int(100 * alpha), 'backbone': bb,
                     'score': sname, 'direction': dv,
                     'merge if distance <=': best,
                     'promised: wrong-link share <= %':
                         round(100 * bound, 1) if bound is not None else None,
                     'delivered wrong-link share %':
                         round(r['mean wrong-link share %'], 1),
                     'held': bool(r['mean wrong-link share %']
                                  <= 100 * alpha),
                     'IDF1 %': round(r['IDF1 %'], 1),
                     'IDs per person': round(r['IDs per person'], 2),
                     'pair precision %': round(r['pair precision %'], 1)})
v2 = pd.DataFrame(rows)
v2.to_csv(f'{OUT}/ilrc_v2_results.csv', index=False)
print('ILRC v2 (person-bootstrap bound, 90% confidence), held-out group')
display(v2)
print('\nMean of both directions:')
display(v2.groupby(['alpha %', 'backbone', 'score'], sort=False)
        [['delivered wrong-link share %', 'IDF1 %', 'IDs per person',
          'pair precision %']].mean().round(1))
print('Promise held in', int(v2.held.sum()), 'of', len(v2), 'cases')
print('Saved to', OUT)
